In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from feature_engine.timeseries.forecasting import LagFeatures, WindowFeatures
from feature_engine.creation import CyclicalFeatures
from feature_engine.datetime import DatetimeFeatures
from feature_engine.imputation import MeanMedianImputer

In [2]:
def load_data():

    # Data lives here.
    filename = "AirQualityUCI_ready.csv"

    # Load data: only the time variable and CO.
    data = pd.read_csv(
        filename,
        usecols=["Date_Time", "CO_sensor"],
        parse_dates=["Date_Time"],
        index_col=["Date_Time"],
    )

    # Sanity: sort index.
    data.sort_index(inplace=True)

    # Reduce data span.
    data = data["2004-04-01":"2005-04-30"]

    # Remove outliers
    data = data.loc[(data["CO_sensor"] > 0)]

    return data

In [3]:
data = load_data()

data.head()

,CO_sensor
Date_Time,
2004-04-04 00:00:00,1224.0
2004-04-04 01:00:00,1215.0
2004-04-04 02:00:00,1115.0
2004-04-04 03:00:00,1124.0
2004-04-04 04:00:00,1028.0


In [4]:
y = data.copy()

In [5]:
dtf = DatetimeFeatures(
    # the input dt variable
    variables="index",
    # the features we want to create
    features_to_extract=[
        "month",
        "week",
        "day_of_week",
        "day_of_month",
        "hour",
        "weekend",
    ],
)

lagf = LagFeatures(
    variables="CO_sensor",  # the input variable
    freq=["1H", "2H"],  # move 1 hr forward
    missing_values="ignore",
)

winf = WindowFeatures(
    variables="CO_sensor",  # the input variable
    window="3H",  # average of 3 previous hours
    freq="1H",  # move 1 hr forward
    missing_values="ignore",
)

cyclicf = CyclicalFeatures(
    # The features we want to transform.
    variables=["month", "hour"],
    # Whether to drop the original features.
    drop_original=False,
)



In [6]:
class ForecastingPipeline(Pipeline):
    
    def __init__(
        self,
        steps, memory=None, verbose=False,
        forecasting_horizon=None,
        freq=None,
    ) -> None:

        self.steps = steps
        self.memory = memory
        self.verbose = verbose
        self.forecasting_horizon = forecasting_horizon
        self.freq=freq
    
    def horizon(self, X):
        
        forecast_start = X.index[-1] + pd.offsets.Hour(1)
        forecast_end = forecast_start + pd.offsets.Hour(self.forecasting_horizon)
        
        horizon = pd.date_range(
            start=forecast_start,
            end=forecast_end,
            freq=self.freq,
        )
        
        return horizon
        

    def forecast(self, X):
        
        X = X.copy()
        
        horizon = self.horizon(X)
        
        X = pd.concat([X, pd.DataFrame(index=horizon, columns=X.columns)], axis=0)
        
        for step in horizon:
            
            input_data = X[X.index < step]
            
            X.loc[step] = self.predict(input_data)[-1]

        return X.loc[horizon]
    
    def transform(self, X):
        Xt = X.copy()
        for _, _, transform in self._iter():
            # need this tweak in case last class is not a
            # transformer
            if hasattr(transform, "transform"):
                Xt = transform.transform(Xt)
        return Xt

In [7]:
pipe = ForecastingPipeline(
    [
        ("datetime_features", dtf),
        ("lagf", lagf),
        ("winf", winf),
        ("Periodic", cyclicf),
        ("drop_data", MeanMedianImputer()),
        ("Lasso", Lasso())
    ],
    forecasting_horizon=5,
    freq="1H",
)

In [8]:
pipe.fit(data, y)

preds = pipe.predict(data)

preds

array([1223.99695266, 1214.99715039, 1114.99934738, ..., 1141.99875419,
       1003.00180801, 1071.00031406])

In [9]:
forecast = pipe.forecast(data)

forecast

,CO_sensor
2005-04-04 15:00:00,1071.000314
2005-04-04 16:00:00,1071.000628
2005-04-04 17:00:00,1071.000942
2005-04-04 18:00:00,1071.001256
2005-04-04 19:00:00,1071.00157
2005-04-04 20:00:00,1071.001884


In [10]:
datat = pipe.transform(data)

datat.head()

,CO_sensor,month,week,day_of_week,day_of_month,hour,weekend,CO_sensor_lag_1H,CO_sensor_lag_2H,CO_sensor_window_3H_mean,month_sin,month_cos,hour_sin,hour_cos
Date_Time,,,,,,,,,,,,,,
2004-04-04 00:00:00,1224.0,4,14,6,4,0,1,1050.0,1050.0,1054.000000,0.866025,-0.5,0.000000,1.000000
2004-04-04 01:00:00,1215.0,4,14,6,4,1,1,1224.0,1050.0,1224.000000,0.866025,-0.5,0.269797,0.962917
2004-04-04 02:00:00,1115.0,4,14,6,4,2,1,1215.0,1224.0,1219.500000,0.866025,-0.5,0.519584,0.854419
2004-04-04 03:00:00,1124.0,4,14,6,4,3,1,1115.0,1215.0,1184.666667,0.866025,-0.5,0.730836,0.682553
2004-04-04 04:00:00,1028.0,4,14,6,4,4,1,1124.0,1115.0,1151.333333,0.866025,-0.5,0.887885,0.460065


In [11]:
pipe.get_feature_names_out(input_features=None)

ValueError: Some features in input_features were not lagged. You can only getthe names of the lagged features with this function.